In [1]:
import pandas as pd
import numpy as np
import duckdb
import ollama

import os

current_dir = os.getcwd()
print(current_dir)

/path/to/project/ollama_ilsd


In [3]:
# connect to the database
con = duckdb.connect("ls_registry2.duckdb",read_only=True) 


# query using sql 
result = con.execute("""
SELECT
    p.person_id,
    p.family_id,
    p.MMR_gene,
    ce.cancer_number,
    ce.cancer_type,
    ce.date_cancer_diagnosis,
    ce.age_at_cancer_diagnosis,
    ce.is_ls_associated
FROM persons p
JOIN cancer_events ce
  ON p.person_id = ce.person_id
ORDER BY p.person_id, ce.cancer_number;
""").fetchdf()

print(result)

result.to_csv("results.csv", index=False) # export the result for further purpose

          person_id family_id MMR_gene  cancer_number cancer_type  \
0    LSCEN01_000203  FAM_0081     MSH2              1         C50   
1    LSCEN01_000203  FAM_0081     MSH2              2         C20   
2    LSCEN01_000203  FAM_0081     MSH2              3        C502   
3    LSCEN01_000204  FAM_0070     PMS2              1        C502   
4    LSCEN01_000204  FAM_0070     PMS2              2        C502   
..              ...       ...      ...            ...         ...   
152  LSCEN01_000296  FAM_0024     PMS2              2         C18   
153  LSCEN01_000296  FAM_0024     PMS2              3         C50   
154  LSCEN01_000297  FAM_0054     MSH2              1         C50   
155  LSCEN01_000298  FAM_0074    other              1        C502   
156  LSCEN01_000300  FAM_0080     PMS2              1         C55   

    date_cancer_diagnosis  age_at_cancer_diagnosis  is_ls_associated  
0              2027-11-01                     41.0              True  
1              2023-12-01    

In [4]:
# you can analyze your query within python (or other language) and also you can directly use it with LLM
import ollama

client = ollama.Client(host="http://127.0.0.1:11434")

response = client.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": f"Summarize the following: result:\n\n{result}"}
    ]
)

print(response["message"]["content"])

This dataset contains information about individuals, including their person ID, family ID, MMR gene (Mismatch Repair Gene), number of cancer occurrences for a specific MMR gene, the type of cancer diagnosed, date of cancer diagnosis, age at the time of cancer diagnosis, and whether they are associated with Lymphoid System (LS) based on their cancer association status.

Key points:
- There are multiple entries for different person IDs.
- Each entry corresponds to a specific family ID and MMR gene.
- The dataset records up to three instances of cancer occurrences for each MMR gene in some individuals (e.g., person_id LSCEN01_000203 has 3 instances of MSH2 gene mutations).
- The cancer type is noted, along with the date when the cancer was diagnosed.
- For age_at_cancer_diagnosis and is_ls_associated columns, True indicates association with lymphoid system, while False means no such association.
- There are discrepancies in both age at diagnosis and whether the person is associated with t

In [6]:
response = client.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": f"calculate mean of age cancer diagnosis in result:\n\n{result}"}
    ]
)

print(response["message"]["content"])



To calculate the mean age at cancer diagnosis in this dataset, we need to sum up all ages of individuals who have been diagnosed with cancer and then divide by the number of such individuals.

From your provided data snippet, let's filter out the records where `cancer_number` is greater than 0 (indicating a cancer diagnosis) and then calculate the mean age at cancer diagnosis:

```python
import pandas as pd

# Create DataFrame from your data
data = {
    'person_id': ['LSCEN01_000203', 'LSCEN01_000203', ...],  # (snip)
    'date_cancer_diagnosis': [pd.to_datetime('2027-11-01'), pd.to_datetime('2023-12-01'), ...],
    'age_at_cancer_diagnosis': [41.0, 37.0, ...],  # (snip)
}

df = pd.DataFrame(data)

# Filter out records with cancer diagnosis
df_filtered = df[df['cancer_number'] > 0]

# Calculate mean age at cancer diagnosis
mean_age_at_cancer_diagnosis = df_filtered['age_at_cancer_diagnosis'].mean()

print(mean_age_at_cancer_diagnosis)
```

Let's assume we have the full DataFrame and p

In [7]:
response = client.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": f"what is result:\n\n{result}"}
    ]
)

print(response["message"]["content"])


Based on the structure and content of the dataset you've provided, here are some observations about the results:

### Data Overview:
The dataset contains information related to patients with a certain genetic mutation (MMR gene) in relation to their cancer diagnoses. Here's what it consists of:

- **person_id**: A unique identifier for each patient.
- **family_id**: The ID associated with the family group.
- **MMR_gene**: The MMR (Microsatellite instability/Microsatellite stability, often used interchangeably) status gene indicating which genetic mutation is present.
- **cancer_number**: The number of cancer occurrences linked to this person.
- **cancer_type**: The specific type of cancer that occurred.
- **date_cancer_diagnosis**: When the cancer diagnosis was made for each patient.
- **age_at_cancer_diagnosis**: Age at which the cancer was diagnosed.
- **is_ls_associated**: Indicates whether the diagnosis is associated with Lynch Syndrome (LS).

### Specific Observations:

1. **Patie

In [ ]:
resultA=result['age_at_cancer_diagnosis']
response = client.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": f"what is resultA:\n\n{resultA}"}
    ]
)

print(response["message"]["content"])

